# BaySeq

## Load and pre-process data

In [28]:
library(dplyr)
library(ggplot2)
library(tidyr)

# Load custom functions into pseudo-namespaces
bs <- new.env()
sys.source("scripts/BaySeq.R", envir = bs)
fn <- new.env()
sys.source("scripts/functions.R", envir = fn)

set.seed(123)

# Select data set
dataname <- "trauma"
dataname <- "Nurses"

# Name of the column designating the center id
center_name <- "hospital"

# Only implemented for linear regression for now
use_local_intercepts <- TRUE

if (startsWith(dataname, "trauma")) {
        data(trauma, package = "BFI")
        data_raw <- trauma
} else if (startsWith(dataname, "Nurses")) {
        data(Nurses, package = "BFI")
        data_raw <- Nurses
} else {
        data_raw <- read.csv(paste0("data/raw/", dataname, ".csv"), row.names = 1)
}

# Shuffling center simulates homogeneous case
if (endsWith(dataname, "_shuffled")) {
        data_raw[[center_name]] <- sample(data[[center_name]])
}

raw_dataname <- dataname
if (use_local_intercepts) {
    dataname <- paste0(dataname, "_", "local_int")
}

data_raw[[center_name]] <- factor(data_raw[[center_name]],
                        levels = sort(unique(data_raw[[center_name]]),
                        decreasing = FALSE))
dataname
head(data_raw)

[1] "Nurses_local_int"

hospital,nurse,age,gender,experien,stress,wardtype,hospsize
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1,36,0,11,7,0,2
1,2,45,0,20,7,0,2
1,3,32,0,7,7,0,2
1,4,57,1,25,6,0,2
1,5,46,1,22,6,0,2
1,6,60,1,22,6,0,2


In [29]:
table(data_raw[[center_name]])
median(table(data_raw[[center_name]]))
sort(unique(data_raw[[center_name]]), decreasing = FALSE)


 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 
36 36 37 38 36 36 40 40 40 40 43 51 52 50 46 40 36 39 40 40 40 36 36 36 36 

[1] 40

[1] 1  2  3  4  5  6  7  8  9  10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25
25 Levels: 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 ... 25

In [30]:
scale_numeric_cols <- function(data) {
  data[] <- lapply(data, function(x) if (is.numeric(x)) scale(x)[, 1] else x)
  data
}

if (startsWith(dataname, "trauma")) {
    data_norm <- data_raw %>%
        # don't need to normalize within group as combined mean and sd can be computed without combining data
        mutate(
            age = scale(age),
            ISS = scale(ISS),
            GCS = scale(GCS)
        )

    model <- mortality ~ sex + age + ISS + GCS
    family <- "binomial" #(link = "logit")

} else if (startsWith(dataname, "Nurses")) {
    data_norm <- data_raw %>%
    mutate(
            age = scale(age),
            experience = scale(experien),
            stress = scale(stress),

            # keep numeric for now; fine if binary 0, 1
            #gender = factor(gender),
            #wardtype = factor(wardtype)
    )
    if (use_local_intercepts) {
        model_no_int <- as.formula(paste("stress ~ 0 + gender + age + experience +", center_name))
        model <- stress ~ gender + age + experience
    } else {
        model <- stress ~ gender + age + experience + wardtype
    }
    family <- "gaussian"

} else {
  stop()
}

covariates <- attr(terms(model), "term.labels")
covariates_local <- covariates[covariates != center_name]
target <- all.vars(model)[1]

if (center_name %in% covariates) {
    data <- data_norm[c(target, covariates)]
} else {
    data <- data_norm[c(center_name, target, covariates)]
}

data_split <- data %>%
  group_by(.data[[center_name]]) %>%
  group_split()

n_centers <- length(data_split)
head(data_split[[1]])

hospital,stress,gender,age,experience
<fct>,"<dbl[,1]>",<dbl>,"<dbl[,1]>","<dbl[,1]>"
1,2.065329,0,-0.5817336,-1.0024484
1,2.065329,0,0.1656757,0.4870737
1,2.065329,0,-0.9139156,-1.6644582
1,1.044405,1,1.1622216,1.3145860
1,1.044405,1,0.2487212,0.8180786
1,1.044405,1,1.4113581,0.8180786


# Using Bayesian Sequential Updating

In [31]:
bstats <- bs$bayseq_prepare(target, covariates, model, data_split, n_centers, use_local_intercepts, center_name)
params_seq <- bs$bayseq_oneshot(bstats, n_centers, use_local_intercepts,
                                covariates, epsilon = 1e-10, center_name)

df_seq <- bs$tidy_results(params_seq, use_local_intercepts)
df_seq

[1] "Bayesian linear regression"
[1] "gender"     "age"        "experience"


Model,Covariate,Value,lower,upper
<chr>,<chr>,<dbl>,<dbl>,<dbl>
BaySeq,gender,-0.4737137,-0.5852068,-0.3622206
BaySeq,age,0.2470454,0.1619393,0.3321516
BaySeq,experience,-0.3570065,-0.4422715,-0.2717415
BaySeq,sigma2,0.6317560,NA,NA


# Using Multivariate Meta-Analysis

In [ ]:
# Fit local GLMs
res_local <- fn$fit_local_glms(data_split, model, target, covariates, center_name)
coef_list <- res_local$coef_list
se_list <- res_local$se_list
cov_list <- res_local$cov_list

In [33]:
# Meta-analyse GLM parameters
results_meta_mv_fe <- fn$fit_mv_meta_fixed(coef_list, cov_list)
results_meta_mv_fe
results_meta_mv_reml <- fn$fit_mv_meta_random(coef_list, cov_list, method="REML")
results_meta_mv_reml

Covariate,Value,lower,upper,Model
<chr>,<dbl>,<dbl>,<dbl>,<chr>
(Intercept),0.4761705,0.3941512,0.5581899,FE
gender,-0.4824995,-0.5783017,-0.3866972,FE
age,0.2643039,0.1915361,0.3370718,FE
experience,-0.3821681,-0.4554922,-0.3088439,FE


Covariate,Value,lower,upper,Model
<chr>,<dbl>,<dbl>,<dbl>,<chr>
(Intercept),0.4528117,0.3550594,0.5505641,REML
gender,-0.4983339,-0.6082770,-0.3883908,REML
age,0.2266000,0.1362747,0.3169252,REML
experience,-0.3869053,-0.4780777,-0.2957329,REML
